# Test fast tree with network builder

Unlike the previous `test_fat_tree_fixed_k.ipynb`, which generated a generic ECLYPSE Infrastructure graph and required manual parsing to classify nodes (as Host or Router) and inject link parameters, this implementation introduces the custom `network_fat_tree generator`. By utilizing `eclypse.builders.network.generator`s, the script directly instantiates a fully configured, native Network object. This eliminates the need for manual topology conversion, significantly streamlining the simulation setup.

In [1]:
import time
import pandas as pd
import random
import numpy as np

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Import of the base Eclypse modules
from eclypse.graph import Application
from eclypse.simulation import Simulation, SimulationConfig
from eclypse.placement.strategies import StaticStrategy

# Import of the Eclypse+NET modules
from eclypse.network import Network, NetworkApplication, PacketGenerationEvent, RoutingEvent, RoutingMetric

# Import the NEW custom topology generator
from eclypse.builders.network.generators import network_fat_tree


def run_eclypse_net_shared(infra: Network, steps: int = 10) -> float:
    """Execute the Eclypse+NET simulation using a pre-built native Network topology."""
    num_nodes = len(infra.nodes)
    app = NetworkApplication(f"Net_App_{num_nodes}")
    mapping = {}

    # Creation of the application nodes and mapping to the infrastructure hosts
    available_hosts = list(infra.hosts)

    for host_id in available_hosts:
        app_name = f"App_{host_id}"
        app.add_node(app_name, cpu=1, ram=1)
        mapping[app_name] = host_id

    # Generation of the traffic
    # The node host[0] sends the traffic to all the other nodes in the network
    if len(available_hosts) > 1:
        source_app = f"App_{available_hosts[0]}"
        for target_node in available_hosts[1:]:
            target_app = f"App_{target_node}"
            app.add_edge(source_app, target_app, packet_size_bytes=1000, avg_packets_per_step=0.1)

    # Setup and execution of the simulation
    packet_evt = PacketGenerationEvent()
    routing_evt = RoutingEvent(step_duration_s=0.001)
    metric = RoutingMetric()

    config = SimulationConfig(
        seed=GLOBAL_SEED,
        max_steps=steps,
        events=[packet_evt, routing_evt, metric],
        path="./results",
        step_every_ms=500,
        include_default_metrics=False,
        report_format="json",
        report_backend="pandas",
        remote=False
    )

    sim = Simulation(infra, simulation_config=config)
    sim.register(app, placement_strategy=StaticStrategy(mapping))

    start_time = time.perf_counter()
    sim.start()
    sim.wait()
    return time.perf_counter() - start_time


k_single = 6

print(f"Generation of the native Network Fat-Tree topology with k={k_single}...")

# Call the custom topology generator to create a Fat-Tree network with the specified parameters
infra_topology = network_fat_tree.get_network_fat_tree(
    k=k_single,
    seed=GLOBAL_SEED,
    bandwidth_mbps=1000,
    length_km=1,
    host_cpu=2,
    host_ram=4
)

total_nodes = len(infra_topology.nodes)

print(f"Total nodes in the topology: {total_nodes}")
print("Starting the simulation...")

# Run the simulation with the generated topology
execution_time = run_eclypse_net_shared(infra_topology, steps=10)

print("-" * 40)
print(f"Simulation completed successfully!")
print(f"Execution time: {execution_time:.4f} seconds")

Generation of the native Network Fat-Tree topology with k=6...
Total nodes in the topology: 99
Starting the simulation...
18:03:41.876 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_0_1, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_0_2, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_0, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_1, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_2, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_0, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_1, 0.1 pkt/step (avg), size 1000B
18:03:41.878 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_2, 0.1 pkt/step (avg), size 1000B
18:03: